# Data and model exploration
- Data source 1 [link](https://www.kaggle.com/datasets/andrewmvd/spotify-playlists)
- Data source 2 [link](https://www.kaggle.com/datasets/devdope/900k-spotify/data)

## Big questions
- How can we validate our model is working?
  - Try to predict if recommendations show on other user's playlist, and that rate
  - Have classmates participate, and have them take 10 or so recommendations and tell us what % they like

In [22]:
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import warnings
import re

# set display options
warnings.filterwarnings("ignore")
pd.set_option('display.max_columns', None)

In [23]:
# read in data
df = pd.read_csv('../../data/spotify_dataset.csv', on_bad_lines='skip')
df.columns=["user","artist","track","playlist"]
df.head()

,user,artist,track,playlist
0,9cc0cfd4d7d7885102480dd99e7a90d6,Elvis Costello,(The Angels Wanna Wear My) Red Shoes,HARD ROCK 2010
1,9cc0cfd4d7d7885102480dd99e7a90d6,Elvis Costello & The Attractions,"(What's So Funny 'Bout) Peace, Love And Unders...",HARD ROCK 2010
2,9cc0cfd4d7d7885102480dd99e7a90d6,Tiffany Page,7 Years Too Late,HARD ROCK 2010
3,9cc0cfd4d7d7885102480dd99e7a90d6,Elvis Costello & The Attractions,Accidents Will Happen,HARD ROCK 2010
4,9cc0cfd4d7d7885102480dd99e7a90d6,Elvis Costello,Alison,HARD ROCK 2010


In [24]:
df2 = pd.read_csv('../../data/spotify_dataset_2.csv')
df2.head()

,Artist(s),song,text,Length,emotion,Genre,Album,Release Date,Key,Tempo,Loudness (db),Time signature,Explicit,Popularity,Energy,Danceability,Positiveness,Speechiness,Liveness,Acousticness,Instrumentalness,Good for Party,Good for Work/Study,Good for Relaxation/Meditation,Good for Exercise,Good for Running,Good for Yoga/Stretching,Good for Driving,Good for Social Gatherings,Good for Morning Routine,Similar Artist 1,Similar Song 1,Similarity Score 1,Similar Artist 2,Similar Song 2,Similarity Score 2,Similar Artist 3,Similar Song 3,Similarity Score 3
0,!!!,Even When the Waters Cold,Friends told her she was better off at the bot...,03:47,sadness,hip hop,Thr!!!er,29th April 2013,D min,105,-6.85db,4/4,No,40,83,71,87,4,16,11,0,0,0,0,0,0,0,0,0,0,Corey Smith,If I Could Do It Again,0.986061,Toby Keith,Drinks After Work,0.983719,Space,Neighbourhood,0.983236
1,!!!,One Girl / One Boy,"Well I heard it, playing soft From a drunken b...",04:03,sadness,hip hop,Thr!!!er,29th April 2013,A# min,117,-5.75db,4/4,No,42,85,70,87,4,32,0,0,0,0,0,0,0,0,0,0,0,Hiroyuki Sawano,BRE@TH//LESS,0.995409,When In Rome,Heaven Knows,0.990905,Justice Crew,Everybody,0.984483
2,!!!,Pardon My Freedom,"Oh my god, did I just say that out loud? Shoul...",05:51,joy,hip hop,Louden Up Now,8th June 2004,A Maj,121,-6.06db,4/4,No,29,89,71,63,8,64,0,20,0,0,0,1,0,0,0,0,0,Ricky Dillard,More Abundantly Medley Live,0.993176,Juliet,Avalon,0.965147,The Jacksons,Lovely One,0.956752
3,!!!,Ooo,[Verse 1] Remember when I called you on the te...,03:44,joy,hip hop,As If,16th October 2015,A min,122,-5.42db,4/4,No,24,84,78,97,4,12,12,0,0,0,0,1,0,0,0,0,0,Eric Clapton,Man Overboard,0.992749,Roxette,Don't Believe In Accidents,0.991494,Tiwa Savage,My Darlin,0.990381
4,!!!,Freedom 15,[Verse 1] Calling me like I got something to s...,06:00,joy,hip hop,As If,16th October 2015,F min,123,-5.57db,4/4,No,30,71,77,70,7,10,4,1,0,0,0,1,0,0,0,0,0,Cibo Matto,Lint Of Love,0.981610,Barrington Levy,Better Than Gold,0.981524,Freestyle,Its Automatic,0.981415


In [25]:
# see if you can join the two datasets and guage success of join

# clean up artist and track names (create new fields for this)
def clean_text(text):
    """
    Drops spaces and lower-cases text
    """
    return str(text).lower().replace(' ', '')
    # if isinstance(text, str): # Check if the input is a string
    #     return re.sub(r'[^\w\s]', '', text)
    # else:
    #     return text # Return the original value if not a string

# df['artist_track'] = (df['artist'].apply(clean_text) + ' ' + 
#                       df['track'].apply(clean_text))
# df2['artist_track'] = (df2['Artist(s)'].apply(clean_text) + ' ' +
#                        df2['song'].apply(clean_text))
# TODO: instead of merging by track/artist, do artist alone (assumption that general things like genre are same and numbers can be averaged)
df['artist_clean'] = (df['artist'].apply(clean_text))
df2['artist_clean'] = (df2['Artist(s)'].apply(clean_text))

In [26]:
df2.head()

,Artist(s),song,text,Length,emotion,Genre,Album,Release Date,Key,Tempo,Loudness (db),Time signature,Explicit,Popularity,Energy,Danceability,Positiveness,Speechiness,Liveness,Acousticness,Instrumentalness,Good for Party,Good for Work/Study,Good for Relaxation/Meditation,Good for Exercise,Good for Running,Good for Yoga/Stretching,Good for Driving,Good for Social Gatherings,Good for Morning Routine,Similar Artist 1,Similar Song 1,Similarity Score 1,Similar Artist 2,Similar Song 2,Similarity Score 2,Similar Artist 3,Similar Song 3,Similarity Score 3,artist_clean
0,!!!,Even When the Waters Cold,Friends told her she was better off at the bot...,03:47,sadness,hip hop,Thr!!!er,29th April 2013,D min,105,-6.85db,4/4,No,40,83,71,87,4,16,11,0,0,0,0,0,0,0,0,0,0,Corey Smith,If I Could Do It Again,0.986061,Toby Keith,Drinks After Work,0.983719,Space,Neighbourhood,0.983236,!!!
1,!!!,One Girl / One Boy,"Well I heard it, playing soft From a drunken b...",04:03,sadness,hip hop,Thr!!!er,29th April 2013,A# min,117,-5.75db,4/4,No,42,85,70,87,4,32,0,0,0,0,0,0,0,0,0,0,0,Hiroyuki Sawano,BRE@TH//LESS,0.995409,When In Rome,Heaven Knows,0.990905,Justice Crew,Everybody,0.984483,!!!
2,!!!,Pardon My Freedom,"Oh my god, did I just say that out loud? Shoul...",05:51,joy,hip hop,Louden Up Now,8th June 2004,A Maj,121,-6.06db,4/4,No,29,89,71,63,8,64,0,20,0,0,0,1,0,0,0,0,0,Ricky Dillard,More Abundantly Medley Live,0.993176,Juliet,Avalon,0.965147,The Jacksons,Lovely One,0.956752,!!!
3,!!!,Ooo,[Verse 1] Remember when I called you on the te...,03:44,joy,hip hop,As If,16th October 2015,A min,122,-5.42db,4/4,No,24,84,78,97,4,12,12,0,0,0,0,1,0,0,0,0,0,Eric Clapton,Man Overboard,0.992749,Roxette,Don't Believe In Accidents,0.991494,Tiwa Savage,My Darlin,0.990381,!!!
4,!!!,Freedom 15,[Verse 1] Calling me like I got something to s...,06:00,joy,hip hop,As If,16th October 2015,F min,123,-5.57db,4/4,No,30,71,77,70,7,10,4,1,0,0,0,1,0,0,0,0,0,Cibo Matto,Lint Of Love,0.981610,Barrington Levy,Better Than Gold,0.981524,Freestyle,Its Automatic,0.981415,!!!


In [27]:
# merge is creating more rows, drop duplicate artist_track/artist_clean from df2
# df2 = df2.drop_duplicates(subset=['artist_track'])
df2 = df2.drop_duplicates(subset=['artist_clean'])

In [28]:
# join datasets and check for missing
print(len(df))
# df = df.merge(df2, how='left', on='artist_track')
df = df.merge(df2, how='left', on='artist_clean')
print(len(df))

12891680
12891680


In [29]:
df.isnull().sum()

user                                    0
artist                              33568
track                                  85
playlist                             1246
artist_clean                            0
Artist(s)                         3580381
song                              3580381
text                              3580381
Length                            3580381
emotion                           3580381
Genre                             3580381
Album                             3581622
Release Date                      3580381
Key                               3580381
Tempo                             3580381
Loudness (db)                     3580381
Time signature                    3580381
Explicit                          3580381
Popularity                        3580381
Energy                            3580381
Danceability                      3580381
Positiveness                      3580381
Speechiness                       3580381
Liveness                          

In [30]:
# artist_track
# 8890167 / 12891680

# artist (spaces removed)
3580381 / 12891680

# artist (characters removed)
# 3824768 / 12891680

0.27772803854889355

In [162]:
df_null = df[df['Artist(s)'].isnull()]['artist'].drop_duplicates()
print(len(df_null))
df_null.head()


260879


2              Tiffany Page
7                  Joe Echo
10             The Breakers
19        Cocktail Slippers
20    Crosby, Stills & Nash
Name: artist, dtype: object

In [149]:
df.head()

,user,artist,track,playlist,artist_track,Artist(s),song,text,Length,emotion,Genre,Album,Release Date,Key,Tempo,Loudness (db),Time signature,Explicit,Popularity,Energy,Danceability,Positiveness,Speechiness,Liveness,Acousticness,Instrumentalness,Good for Party,Good for Work/Study,Good for Relaxation/Meditation,Good for Exercise,Good for Running,Good for Yoga/Stretching,Good for Driving,Good for Social Gatherings,Good for Morning Routine,Similar Artist 1,Similar Song 1,Similarity Score 1,Similar Artist 2,Similar Song 2,Similarity Score 2,Similar Artist 3,Similar Song 3,Similarity Score 3
0,9cc0cfd4d7d7885102480dd99e7a90d6,Elvis Costello,(The Angels Wanna Wear My) Red Shoes,HARD ROCK 2010,elviscostello (theangelswannawearmy)redshoes,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,9cc0cfd4d7d7885102480dd99e7a90d6,Elvis Costello & The Attractions,"(What's So Funny 'Bout) Peace, Love And Unders...",HARD ROCK 2010,elviscostello&theattractions (what'ssofunny'bo...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,9cc0cfd4d7d7885102480dd99e7a90d6,Tiffany Page,7 Years Too Late,HARD ROCK 2010,tiffanypage 7yearstoolate,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,9cc0cfd4d7d7885102480dd99e7a90d6,Elvis Costello & The Attractions,Accidents Will Happen,HARD ROCK 2010,elviscostello&theattractions accidentswillhappen,Elvis Costello & The Attractions,Accidents Will Happen,"[Verse 1] Oh, I just don't know where to begin...",03:01,sadness,"folk,country,new wave",Armed Forces (Super Deluxe Edition),5th January 1979,C Maj,120.0,-11.12db,4/4,No,37.0,60.0,61.0,74.0,3.0,28.0,4.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,Adam Green,Friends of Mine,0.981811,MAY-A,Time I Love To Waste,0.979441,KOPPS,Dumb,0.975395
4,9cc0cfd4d7d7885102480dd99e7a90d6,Elvis Costello,Alison,HARD ROCK 2010,elviscostello alison,Elvis Costello,Alison,"[Verse 1] Oh, it's so funny to be seeing you a...",03:24,surprise,"folk,country,new wave",My Aim Is True,22nd July 1977,C# min,177.0,-10.79db,4/4,No,51.0,32.0,56.0,38.0,4.0,11.0,74.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,Alison Krauss & Union Station,When You Say Nothing At All,0.995125,Restless Heart,Ill Still Be Loving You,0.994933,Cavetown,888,0.992878


In [145]:
df.describe()

,Tempo,Popularity,Energy,Danceability,Positiveness,Speechiness,Liveness,Acousticness,Instrumentalness,Good for Party,Good for Work/Study,Good for Relaxation/Meditation,Good for Exercise,Good for Running,Good for Yoga/Stretching,Good for Driving,Good for Social Gatherings,Good for Morning Routine,Similarity Score 1,Similarity Score 2,Similarity Score 3
count,4.001513e+06,4.001513e+06,4.001513e+06,4.001513e+06,4.001513e+06,4.001513e+06,4.001513e+06,4.001513e+06,4.001513e+06,4.001513e+06,4.001513e+06,4.001513e+06,4.001513e+06,4.001513e+06,4.001513e+06,4.001513e+06,4.001513e+06,4.001513e+06,4.001513e+06,4.001513e+06,4.001513e+06
mean,1.226598e+02,4.712917e+01,6.781739e+01,5.387383e+01,4.870788e+01,6.786820e+00,1.887197e+01,2.115269e+01,8.385761e+00,1.229028e-01,6.479149e-02,2.474214e-02,1.858707e-01,4.718515e-02,1.695184e-02,3.980095e-02,2.485785e-02,5.967793e-02,9.841468e-01,9.784058e-01,9.751931e-01
std,2.804027e+01,1.895644e+01,2.245538e+01,1.590421e+01,2.438652e+01,7.028922e+00,1.507258e+01,2.812374e+01,2.071285e+01,3.283256e-01,2.461576e-01,1.553383e-01,3.890023e-01,2.120347e-01,1.290910e-01,1.954913e-01,1.556918e-01,2.368892e-01,1.435110e-02,1.546783e-02,1.623362e-02
min,3.500000e+01,0.000000e+00,0.000000e+00,6.000000e+00,0.000000e+00,2.000000e+00,1.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,2.656466e-03,2.647312e-03,2.646671e-03
25%,1.010000e+02,3.300000e+01,5.300000e+01,4.300000e+01,2.900000e+01,3.000000e+00,1.000000e+01,1.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,9.776131e-01,9.716460e-01,9.680626e-01
50%,1.210000e+02,4.600000e+01,7.200000e+01,5.400000e+01,4.800000e+01,4.000000e+00,1.300000e+01,6.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,9.864086e-01,9.811607e-01,9.782829e-01
75%,1.400000e+02,6.200000e+01,8.600000e+01,6.500000e+01,6.800000e+01,7.000000e+00,2.400000e+01,3.300000e+01,2.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,9.937276e-01,9.880088e-01,9.854569e-01
max,2.000000e+02,9.900000e+01,1.000000e+02,9.900000e+01,9.900000e+01,9.700000e+01,1.000000e+02,1.000000e+02,1.000000e+02,1.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00


# Data exploration

## Questions

- How many artists do we have?
  - 289,821
- How many users do we have?
  - 15,918
- How many playlists do users have?
  - 15, on average
- What are the top 5 most popular artists?
  - Coldplay
  - Daft Punk
  - Rihanna
  - David Guetta
  - Calvin Harris
- What are the top 5 most popular songs?
  - m83 midnightcity
  - daftpunk getlucky-radioedit
  - imaginedragons radioactive
  - ofmonstersandmen littletalks
  - avicii wakemeup

In [117]:
# number of artists
num_artists = df['artist'].nunique()
num_artists

289821

In [118]:
# number of users
num_users = df['user'].nunique()
num_users

15918

In [119]:
# avg number of playlists per user
playlist_user = df[['user', 'playlist']].drop_duplicates()
playlist_user.head()

,user,playlist
0,9cc0cfd4d7d7885102480dd99e7a90d6,HARD ROCK 2010
67,9cc0cfd4d7d7885102480dd99e7a90d6,IOW 2012
104,07f0fc3be95dcd878966b1f9572ff670,2080
114,07f0fc3be95dcd878966b1f9572ff670,C418
148,07f0fc3be95dcd878966b1f9572ff670,Chill out


In [120]:
# user playlist counts
playlist_user_counts = playlist_user.groupby('user')['playlist'].count()
playlist_user_counts.mean()

14.562382208820203

# Data preprocessing

In [ ]:
# create combined artist/track feature
def clean_text(text):
    """
    Drops spaces and lower-cases text
    """
    return str(text).lower().replace(' ', '')

df['artist_track'] = (df['artist'].apply(clean_text) + ' ' + 
                      df['track'].apply(clean_text))
df.head()

,user,artist,track,playlist,artist_track
0,9cc0cfd4d7d7885102480dd99e7a90d6,Elvis Costello,(The Angels Wanna Wear My) Red Shoes,HARD ROCK 2010,elviscostello (theangelswannawearmy)redshoes
1,9cc0cfd4d7d7885102480dd99e7a90d6,Elvis Costello & The Attractions,"(What's So Funny 'Bout) Peace, Love And Unders...",HARD ROCK 2010,elviscostello&theattractions (what'ssofunny'bo...
2,9cc0cfd4d7d7885102480dd99e7a90d6,Tiffany Page,7 Years Too Late,HARD ROCK 2010,tiffanypage 7yearstoolate
3,9cc0cfd4d7d7885102480dd99e7a90d6,Elvis Costello & The Attractions,Accidents Will Happen,HARD ROCK 2010,elviscostello&theattractions accidentswillhappen
4,9cc0cfd4d7d7885102480dd99e7a90d6,Elvis Costello,Alison,HARD ROCK 2010,elviscostello alison


In [122]:
# run user counts for each artist_track and each artist - can be used to measure song popularity
track_user_count = df.groupby('artist_track')['user'].count()
track_user_count = pd.DataFrame(track_user_count).reset_index()
track_user_count.columns = ['artist_track', 'count']
track_user_count.sort_values('count', ascending=False).head()

,artist_track,count
1462170,m83 midnightcity,2609
517563,daftpunk getlucky-radioedit,2341
1049832,imaginedragons radioactive,2336
1764670,ofmonstersandmen littletalks,2263
181773,avicii wakemeup,2242


In [123]:
# find most popular artists (count of users with artist in playlist)
artist_user_count = df[['user', 'artist']].drop_duplicates() \
    .groupby('artist')['user'].count()
artist_user_count = pd.DataFrame(artist_user_count).reset_index()
artist_user_count.columns = ['artist', 'count']
artist_user_count.sort_values('count', ascending=False).head()

,artist,count
50551,Coldplay,4645
59037,Daft Punk,4631
210322,Rihanna,4092
63007,David Guetta,3861
39657,Calvin Harris,3732


In [124]:
# create artist_popularity and track_popularity features
# artist popularity
artist_user_count['artist_popularity'] = artist_user_count['count'] / num_users
artist_user_count = artist_user_count[['artist', 'artist_popularity']]
artist_user_count.sort_values('artist_popularity', ascending=False).head()

,artist,artist_popularity
50551,Coldplay,0.291808
59037,Daft Punk,0.290929
210322,Rihanna,0.257067
63007,David Guetta,0.242556
39657,Calvin Harris,0.234452


In [125]:
# track popularity
track_user_count['track_popularity'] = track_user_count['count'] / num_users
track_user_count = track_user_count[['artist_track', 'track_popularity']]
track_user_count.sort_values('track_popularity', ascending=False).head()

,artist_track,track_popularity
1462170,m83 midnightcity,0.163903
517563,daftpunk getlucky-radioedit,0.147066
1049832,imaginedragons radioactive,0.146752
1764670,ofmonstersandmen littletalks,0.142166
181773,avicii wakemeup,0.140847


In [126]:
# merge both with main dataset
print(len(df))
df = df.merge(track_user_count, how='left', on='artist_track')
df = df.merge(artist_user_count, how='left', on='artist')
print(len(df))
df.head()

12891680
12891680


,user,artist,track,playlist,artist_track,track_popularity,artist_popularity
0,9cc0cfd4d7d7885102480dd99e7a90d6,Elvis Costello,(The Angels Wanna Wear My) Red Shoes,HARD ROCK 2010,elviscostello (theangelswannawearmy)redshoes,0.004837,0.041965
1,9cc0cfd4d7d7885102480dd99e7a90d6,Elvis Costello & The Attractions,"(What's So Funny 'Bout) Peace, Love And Unders...",HARD ROCK 2010,elviscostello&theattractions (what'ssofunny'bo...,0.005403,0.029212
2,9cc0cfd4d7d7885102480dd99e7a90d6,Tiffany Page,7 Years Too Late,HARD ROCK 2010,tiffanypage 7yearstoolate,0.000063,0.000063
3,9cc0cfd4d7d7885102480dd99e7a90d6,Elvis Costello & The Attractions,Accidents Will Happen,HARD ROCK 2010,elviscostello&theattractions accidentswillhappen,0.004774,0.029212
4,9cc0cfd4d7d7885102480dd99e7a90d6,Elvis Costello,Alison,HARD ROCK 2010,elviscostello alison,0.013632,0.041965


## Model research links

- https://365datascience.com/tutorials/how-to-build-recommendation-system-in-python/


## Initial modeling steps

### MVP model
- combine artist and track to create unique ID for each song
  - vectorize track/artist names with count vectorizer (lengths are similar)

### User similarity
- Look into collaborative filtering, factoring in user IDs
  
### Exploratory
- extract information from playlist names
  - vectorize playlist names to get same "feeling"

In [127]:
# MVP model (only factors in artist-track info)
df_mvp = df[['artist_track']].copy().drop_duplicates()
# sample a smaller amount of data to avoid kernel crash
df_mvp = df_mvp.sample(n=15000, replace=False, random_state=42)
vectorizer = CountVectorizer()
vectorized = vectorizer.fit_transform(df_mvp['artist_track'])
similarities = cosine_similarity(vectorized)

In [128]:
similarities = pd.DataFrame(similarities, 
                            columns=df_mvp['artist_track'], 
                            index=df_mvp['artist_track']).reset_index()
similarities.head()

artist_track,artist_track,rickbraun cadillacslim,robertwells music,ninorota ilpellegrinaggio,whitneyhouston it'snotrightbutit'sokay-club69clubmix,2unlimited unlimitedmegajam,ericdolphy softlyasinamorningsunrise-live,ghostbeach miracle(gigameshremix),"chetbaker,stangetz 3+1=5",bryanferry let'ssticktogether,...,prissyclerks blast-offgirls,johnbarry&hisorchestra beatgirl(maintitle),peetahmorgan byebye,bossfight underskerimörkagränder,becalmhoncho whatwehavemade,greenpointorchestra 6-utra,peggyleewithdavebarbourandhisorchestra i'mgonnawashthatmanrightoutofmyhair,robertschumann carnavalop.9-06,ogmaco beenthuggin,2pac theydon'tgiveafuckaboutus
0,rickbraun cadillacslim,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,robertwells music,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,ninorota ilpellegrinaggio,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,whitneyhouston it'snotrightbutit'sokay-club69c...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,2unlimited unlimitedmegajam,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [129]:
input = "2pac theydon'tgiveafuckaboutus"
recommendations = pd.DataFrame(similarities.nlargest(11, input)['artist_track'])
recommendations = recommendations[recommendations['artist_track']!=input]
print(recommendations)

                                            artist_track
2938                                     2pac somuchpain
5016                                   2pac whatzyaphone
9444                         2pac wondawhytheycallubytch
9979                             2pac shortywannabeathug
11112               2pac/snoopdogg 2ofamerikazmostwanted
6200                               2pac pac'slife(remix)
7817                      paulwall theydon'tknow(ft.mike
7026             earth,wind&fire theydon'tsee-remastered
11394  childishgambino theydon'tlikeme(ft.chancethera...
0                                 rickbraun cadillacslim


## MVP model notes

- only factors in name similarity (same artist, titles with same words)
- while artist matches could be accurate, they are also obvious